# FoodMkt_R — Reusable Template

Drop in a new food / CPG / restaurant / agri snapshot and walk: clean → outlier-aware EDA → neighbor averages → flag index → screen → path/MA chart.

Replace every `TODO`.


In [ ]:
library(ggplot2)
library(plyr)
library(reshape2)
library(zoo)
theme_set(theme_minimal())

snap_path <- "data/foodviz.csv"          # TODO
id_cols <- c("Ticker", "Company", "Sector", "Industry", "Country")
val_vars <- c("Price", "P.E", "PEG", "P.S", "P.B")
idx_cut <- 8
price_lo <- 20; price_hi <- 100
max_de <- 1; max_beta <- 1.5; max_inst <- 30
country_keep <- "USA"
outlier_ticker <- "GODIVA"               # TODO: or NA

clean_numeric <- function(s) {
  as.numeric(gsub("%|\\$|,|\\)|\\(", "", s))
}

raw <- read.csv(snap_path, stringsAsFactors = FALSE, check.names = FALSE)
stopifnot(all(id_cols %in% names(raw)))
num_idx <- which(!names(raw) %in% id_cols)
raw[num_idx] <- lapply(raw[num_idx], clean_numeric)
names(raw) <- make.names(names(raw))
if (!is.na(outlier_ticker)) raw <- subset(raw, Ticker != outlier_ticker)

melt_avg <- function(df, ids, vars, prefix) {
  m <- melt(df, id = ids)
  m <- subset(m, variable %in% vars)
  m <- na.omit(m)
  m$value <- as.numeric(m$value)
  form <- as.formula(paste(paste(ids, collapse = " + "), "~ variable"))
  wide <- dcast(m, form, mean)
  for (v in vars) {
    if (v %in% names(wide)) names(wide)[names(wide) == v] <- paste0(prefix, v)
  }
  wide
}

savg <- melt_avg(raw, "Sector", val_vars, "S_")
iavg <- melt_avg(raw, c("Sector", "Industry"), val_vars, "I_")
df <- merge(raw, savg, by = "Sector")
df <- merge(df, iavg, by = c("Sector", "Industry"))

for (v in val_vars) {
  df[[paste0("Sunder_", v)]] <- as.integer(df[[v]] < df[[paste0("S_", v)]])
  df[[paste0("Iunder_", v)]] <- as.integer(df[[v]] < df[[paste0("I_", v)]])
}
flag_cols <- grep("under_", names(df), value = TRUE)
df$RelValIndex <- rowSums(df[flag_cols], na.rm = TRUE)

passers <- subset(
  df,
  Country == country_keep &
    Price > price_lo & Price < price_hi &
    RelValIndex >= idx_cut
)
nrow(passers)
head(passers[order(-passers$RelValIndex), c("Ticker", "Company", "Sector", "RelValIndex", "Price")])

## Checklist when the vendor file changes

1. `names(raw)` before and after `make.names`.
2. Confirm `%` / `$` / commas still exist.
3. Revisit `outlier_ticker` (new luxury listing, dual-class share, ADR).
4. Re-tune `idx_cut` so a human can read the passer list.
5. Write four audience paragraphs before mailing category managers.
